In [2]:
%pip install kagglehub


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [6]:
# imports
import pandas as pd

In [4]:
import kagglehub
import shutil
import os

# Download (goes to kagglehub's cache first)
path = kagglehub.dataset_download("artemkabseu/financial-transactions-dataset-expenses-and-income")

# Create a folder in your current directory
dest_folder = os.path.join(os.getcwd(), "data")
os.makedirs(dest_folder, exist_ok=True)

# Copy everything from the cache path into your folder
for filename in os.listdir(path):
    src = os.path.join(path, filename)
    dst = os.path.join(dest_folder, filename)
    shutil.copy2(src, dst)

print(f"Saved to: {dest_folder}")
print(os.listdir(dest_folder))


Saved to: /Users/heiditam/Documents/variance_analysis_agent/data
['Income_clean.csv', 'Expenses_clean.csv']


In [7]:
expenses = pd.read_csv('data/Expenses_clean.csv')

In [8]:
expenses.head()

,date_time,category,account,amount,currency,tags
0,2025-11-30 00:00:00,Health,acct_1,114.0,BYN,tag_1
1,2025-11-29 00:00:00,Food,acct_1,5.0,BYN,tag_1
2,2025-11-27 00:00:00,Public transport,acct_2,1.0,BYN,tag_1
3,2025-11-27 00:00:00,Cafe,acct_1,10.0,BYN,tag_1
4,2025-11-27 00:00:00,Public transport,acct_2,1.0,BYN,tag_1


In [12]:
income = pd.read_csv('data/Income_clean.csv')
income

,date_time,category,account,amount,currency,tags
0,2025-11-29 00:00:00,Job,acct_1,49.0,BYN,tag_1
1,2025-11-29 00:00:00,Job,acct_1,18.0,BYN,tag_2
2,2025-11-29 00:00:00,Job,acct_1,32.0,BYN,tag_3
3,2025-11-29 00:00:00,Job,acct_1,109.0,BYN,tag_4
4,2025-11-28 00:00:00,Second work,acct_1,132.0,BYN,2th_work
...,...,...,...,...,...,...
344,2025-01-07 00:00:00,Cashback,acct_1,7.0,BYN,tag_5
345,2025-01-05 00:00:00,Second work,acct_1,78.0,BYN,2th_work
346,2025-01-04 00:00:00,Gift,acct_1,51.0,BYN,tag_5
347,2025-01-04 00:00:00,Gift,acct_1,77.0,BYN,tag_5


In [19]:
joined_df = pd.concat([expenses, income], axis=0, keys=['expenses', 'income'])
joined_df = joined_df.reset_index(level=0).rename(columns={'level_0': 'type'})
joined_df

,type,date_time,category,account,amount,currency,tags
0,expenses,2025-11-30 00:00:00,Health,acct_1,114.0,BYN,tag_1
1,expenses,2025-11-29 00:00:00,Food,acct_1,5.0,BYN,tag_1
2,expenses,2025-11-27 00:00:00,Public transport,acct_2,1.0,BYN,tag_1
3,expenses,2025-11-27 00:00:00,Cafe,acct_1,10.0,BYN,tag_1
4,expenses,2025-11-27 00:00:00,Public transport,acct_2,1.0,BYN,tag_1
...,...,...,...,...,...,...,...
344,income,2025-01-07 00:00:00,Cashback,acct_1,7.0,BYN,tag_5
345,income,2025-01-05 00:00:00,Second work,acct_1,78.0,BYN,2th_work
346,income,2025-01-04 00:00:00,Gift,acct_1,51.0,BYN,tag_5
347,income,2025-01-04 00:00:00,Gift,acct_1,77.0,BYN,tag_5


In [20]:
# fix data types and add a period column
joined_df['date_time'] = pd.to_datetime(joined_df['date_time'])
joined_df['period'] = joined_df['date_time'].dt.to_period('M').astype(str)  # e.g. '2025-11'

# mark expenses as negative
joined_df['net_amount'] = joined_df.apply(
    lambda r: -r['amount'] if r['type'] == 'expenses' else r['amount'], axis=1
)

# build monthly summary view
summary = joined_df.groupby(['period', 'account', 'category', 'type'])['net_amount'].sum().reset_index()

summary

,period,account,category,type,net_amount
0,2025-01,acct_1,Cafe,expenses,-70.0
1,2025-01,acct_1,Cashback,income,13.0
2,2025-01,acct_1,Food,expenses,-49.0
3,2025-01,acct_1,Gift,income,128.0
4,2025-01,acct_1,Gifts,expenses,-170.0
...,...,...,...,...,...
217,2025-11,acct_2,Gift,income,3.0
218,2025-11,acct_2,Public transport,expenses,-45.0
219,2025-11,acct_3,Cafe,expenses,-37.0
220,2025-11,acct_3,Food,expenses,-10.0


In [21]:
print(joined_df['period'].value_counts().sort_index())
print(joined_df.groupby('type')['account'].nunique())

period
2025-01     91
2025-02     87
2025-03    131
2025-04    113
2025-05    107
2025-06    125
2025-07    110
2025-08    119
2025-09    150
2025-10    145
2025-11    109
Name: count, dtype: int64
type
expenses    3
income      5
Name: account, dtype: int64


In [23]:
category_summary = joined_df.groupby(['period', 'type', 'category'])['net_amount'].sum().reset_index()

# pivot so each period is a column, easy to diff
pivot = category_summary.pivot_table(
    index=['type', 'category'], columns='period', values='net_amount', fill_value=0
)

# pick the two periods you're comparing
periods = sorted(pivot.columns)
prior, current = periods[-2], periods[-1]

pivot['delta'] = pivot[current] - pivot[prior]
pivot['pct_change'] = pivot['delta'] / pivot[prior].replace(0, pd.NA)

variance_ranked = pivot.sort_values('delta', key=abs, ascending=False)
variance_ranked[[prior, current, 'delta', 'pct_change']].head(10)

period                                 2025-10  2025-11   delta pct_change
type     category                                                         
income   Debt return / Borrowed money   1370.0     66.0 -1304.0  -0.951825
expenses Loan given                     -992.0    -23.0   969.0  -0.976815
         Clothes                           0.0   -299.0  -299.0       <NA>
income   Second work                     549.0    265.0  -284.0  -0.517304
         Gift                            404.0    197.0  -207.0  -0.512376
         Job                             670.0    789.0   119.0   0.177612
expenses Gifts                          -174.0    -79.0    95.0  -0.545977
         Food                           -108.0    -35.0    73.0  -0.675926
         Bought for myself               -64.0      0.0    64.0       -1.0
         Health                         -199.0   -150.0    49.0  -0.246231

In [24]:
top_type, top_category = variance_ranked.index[0]  # e.g. ('income', 'Gift')

drill = joined_df[
    (joined_df['type'] == top_type) &
    (joined_df['category'] == top_category) &
    (joined_df['period'] == current)
]

tag_breakdown = drill.groupby('tags')['net_amount'].sum().sort_values(ascending=False)
tag_pct = tag_breakdown / tag_breakdown.sum()

print(tag_breakdown)
print(tag_pct)

tags
tag_5    66.0
Name: net_amount, dtype: float64
tags
tag_5    1.0
Name: net_amount, dtype: float64


In [25]:
top_tags = tag_pct.head(3)
concentration_pct = top_tags.sum()

print(f"{concentration_pct:.0%} of the change in {top_category} ({top_type}) "
      f"came from tags: {', '.join(top_tags.index)}")

100% of the change in Debt return / Borrowed money (income) came from tags: tag_5


In [30]:
import pandas as pd

# --- 1. Clean dtypes ---
joined_df['date_time'] = pd.to_datetime(joined_df['date_time'])
joined_df['period'] = joined_df['date_time'].dt.to_period('M').astype(str)

# --- 2. Signed amount (expenses negative) ---
joined_df['net_amount'] = joined_df.apply(
    lambda r: -r['amount'] if r['type'] == 'expenses' else r['amount'], axis=1
)

# --- 3. Monthly summary by type/category ---
category_summary = joined_df.groupby(['period', 'type', 'category'])['net_amount'].sum().reset_index()
pivot = category_summary.pivot_table(
    index=['type', 'category'], columns='period', values='net_amount', fill_value=0
)

periods = sorted(pivot.columns)
prior, current = periods[0], periods[-1]

# --- 4. Delta + safe pct_change (no NaN/inf, explicit is_new flag) ---
pivot['delta'] = pivot[current] - pivot[prior]
pivot['is_new'] = pivot[prior] == 0
pivot['pct_change'] = pivot.apply(
    lambda r: (r['delta'] / r[prior]) if r[prior] != 0 else None, axis=1
)

# --- 5. Transaction-count guardrail (avoid 1-transaction "stories") ---
txn_counts = joined_df[joined_df['period'] == current].groupby(['type', 'category']).size()
pivot['n_transactions'] = pivot.index.map(txn_counts).fillna(0).astype(int)

# --- 6. Rank only candidates backed by real volume ---
MIN_TRANSACTIONS = 3
candidates = pivot[pivot['n_transactions'] >= MIN_TRANSACTIONS].copy()
candidates = candidates.sort_values('delta', key=abs, ascending=False)

print(candidates[[prior, current, 'delta', 'pct_change', 'is_new', 'n_transactions']])

# --- 7. Drill-down + evidence dict for the top valid candidate ---
if len(candidates) > 0:
    top_type, top_category = candidates.index[0]

    drill = joined_df[
        (joined_df['type'] == top_type) &
        (joined_df['category'] == top_category) &
        (joined_df['period'] == current)
    ]

    tag_breakdown = drill.groupby('tags')['net_amount'].sum().sort_values(key=abs, ascending=False)
    tag_pct = (tag_breakdown / tag_breakdown.sum()).abs()
    top_tags = tag_pct.head(3)

    row = candidates.iloc[0]
    evidence = {
        "type": top_type,
        "category": top_category,
        "prior_period_value": float(row[prior]),
        "current_period_value": float(row[current]),
        "delta": float(row['delta']),
        "pct_change": None if pd.isna(row['pct_change']) else float(row['pct_change']),
        "is_new": bool(row['is_new']),
        "n_transactions": int(row['n_transactions']),
        "top_drivers": [
            {"tag": tag, "amount": float(tag_breakdown[tag]), "share_of_change": float(pct)}
            for tag, pct in top_tags.items()
        ]
    }

    print("\nEvidence:")
    print(evidence)
else:
    print("No candidates meet the minimum transaction threshold — lower MIN_TRANSACTIONS or check data coverage.")

period                     2025-01  2025-11  delta  pct_change  is_new  \
type     category                                                        
income   Job                 506.0    789.0  283.0    0.559289   False   
expenses Cafe                -70.0   -173.0 -103.0    1.471429   False   
income   Gift                128.0    197.0   69.0    0.539062   False   
expenses Food                -86.0    -35.0   51.0   -0.593023   False   
         Public transport    -33.0    -45.0  -12.0    0.363636   False   

period                     n_transactions  
type     category                          
income   Job                           20  
expenses Cafe                          25  
income   Gift                           3  
expenses Food                           7  
         Public transport              41  

Evidence:
{'type': 'income', 'category': 'Job', 'prior_period_value': 506.0, 'current_period_value': 789.0, 'delta': 283.0, 'pct_change': 0.5592885375494071, 'is_new': Fals

In [31]:
# is new is a boolean flag marking categories that had zero activity in the prior period but have activity in the current one
candidates[[prior, current, 'delta', 'pct_change', 'is_new', 'n_transactions']]

period                     2025-01  2025-11  delta  pct_change  is_new  \
type     category                                                        
income   Job                 506.0    789.0  283.0    0.559289   False   
expenses Cafe                -70.0   -173.0 -103.0    1.471429   False   
income   Gift                128.0    197.0   69.0    0.539062   False   
expenses Food                -86.0    -35.0   51.0   -0.593023   False   
         Public transport    -33.0    -45.0  -12.0    0.363636   False   

period                     n_transactions  
type     category                          
income   Job                           20  
expenses Cafe                          25  
income   Gift                           3  
expenses Food                           7  
         Public transport              41

In [32]:
candidates

period                     2025-01  2025-02  2025-03  2025-04  2025-05  \
type     category                                                        
income   Job                 506.0    733.0    750.0    516.0    665.0   
expenses Cafe                -70.0   -149.0   -109.0   -111.0   -116.0   
income   Gift                128.0    117.0     29.0      6.0     95.0   
expenses Food                -86.0    -53.0    -37.0    -54.0    -33.0   
         Public transport    -33.0    -27.0    -39.0    -46.0    -42.0   

period                     2025-06  2025-07  2025-08  2025-09  2025-10  \
type     category                                                        
income   Job                 997.0    456.0    788.0    938.0    670.0   
expenses Cafe               -130.0    -60.0   -218.0   -109.0   -179.0   
income   Gift                110.0     43.0     17.0   3489.0    404.0   
expenses Food               -162.0   -252.0   -175.0   -543.0   -108.0   
         Public transport    -42.0    -30.0    -59.0    -73.0    -78.0   

period                     2025-11  delta  is_new  pct_change  n_transactions  
type     category                                                              
income   Job                 789.0  283.0   False    0.559289              20  
expenses Cafe               -173.0 -103.0   False    1.471429              25  
income   Gift                197.0   69.0   False    0.539062               3  
expenses Food                -35.0   51.0   False   -0.593023               7  
         Public transport    -45.0  -12.0   False    0.363636              41

In [37]:
joined_df.to_csv("data/joined_transactions.csv", index=False)

In [39]:
candidates.reset_index().to_csv("data/candidates.csv", index=False)

In [41]:
drill.to_csv('data/drill.csv',index=False)

In [34]:
candidates.index

MultiIndex([(  'income',              'Job'),
            ('expenses',             'Cafe'),
            (  'income',             'Gift'),
            ('expenses',             'Food'),
            ('expenses', 'Public transport')],
           names=['type', 'category'])

In [33]:
evidence

{'type': 'income',
 'category': 'Job',
 'prior_period_value': 506.0,
 'current_period_value': 789.0,
 'delta': 283.0,
 'pct_change': 0.5592885375494071,
 'is_new': False,
 'n_transactions': 20,
 'top_drivers': [{'tag': 'tag_4',
   'amount': 349.0,
   'share_of_change': 0.4423320659062104},
  {'tag': 'tag_1', 'amount': 174.0, 'share_of_change': 0.22053231939163498},
  {'tag': 'tag_3', 'amount': 140.0, 'share_of_change': 0.17743979721166034}]}

In [35]:
drill = joined_df[
    (joined_df['type'] == top_type) &
    (joined_df['category'] == top_category) &
    (joined_df['period'] == current)
]

In [36]:
drill

,type,date_time,category,account,amount,currency,tags,period,net_amount
0,income,2025-11-29,Job,acct_1,49.0,BYN,tag_1,2025-11,49.0
1,income,2025-11-29,Job,acct_1,18.0,BYN,tag_2,2025-11,18.0
2,income,2025-11-29,Job,acct_1,32.0,BYN,tag_3,2025-11,32.0
3,income,2025-11-29,Job,acct_1,109.0,BYN,tag_4,2025-11,109.0
6,income,2025-11-25,Job,acct_1,18.0,BYN,tag_1,2025-11,18.0
7,income,2025-11-22,Job,acct_1,44.0,BYN,tag_1,2025-11,44.0
8,income,2025-11-22,Job,acct_1,10.0,BYN,tag_2,2025-11,10.0
9,income,2025-11-22,Job,acct_1,11.0,BYN,tag_3,2025-11,11.0
10,income,2025-11-20,Job,acct_1,133.0,BYN,tag_4,2025-11,133.0
11,income,2025-11-18,Job,acct_1,17.0,BYN,tag_1,2025-11,17.0
